In [4]:
# install or upgrade required packages
%pip install mlflow==3.3.1 langchain langchain-community langchain-openai python-dotenv sentence-transformers faiss-cpu --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 28.9 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
#Imports & setup
import os
import mlflow
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
# Ensure you have your Azure credentials in a .env file in the project root
load_dotenv('../.env')

# --- Configuration ---
MLFLOW_TRACKING_URI = "http://mlflow:5000"
EXPERIMENT_NAME = "Building-Companion-Prompt-Engineering"
VECTOR_STORE_PATH = "../artifacts/vector_store"

# --- MLflow Setup ---
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ MLflow and libraries are ready.")
print(f"Tracking experiments in: '{EXPERIMENT_NAME}'")

✅ MLflow and libraries are ready.
Tracking experiments in: 'Building-Companion-Prompt-Engineering'


In [6]:
# load the vector store
print("Loading embedding model...")
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = HuggingFaceEmbeddings(model_name=model_name)

print(f"Loading vector store from: {VECTOR_STORE_PATH}")
db = FAISS.load_local(VECTOR_STORE_PATH, embedder, allow_dangerous_deserialization=True)
retriever = db.as_retriever()

print("✅ Knowledge base is loaded and ready.")

Loading embedding model...
Loading vector store from: ../artifacts/vector_store
✅ Knowledge base is loaded and ready.


In [7]:
# Define the prompt templates
prompt_templates = {
    "legal_expert": """
    You are an AI assistant specialized in Portuguese building regulations based on the provided document.
    Use the following pieces of context to answer the question at the end.
    If you don't know the answer from the context, just say that you don't know, don't try to make up an answer.
    Provide a concise and direct answer based strictly on the provided text. Cite the article number if possible.

    Context: {context}

    Question: {question}

    Answer (in Portuguese):
    """,
    "homeowner_guide": """
    You are a helpful AI assistant for homeowners in Portugal.
    Use the information from the provided legal document to answer the user's question in a simple, easy-to-understand way.
    Explain the key points without using complex legal jargon.

    Based on the regulations: {context}

    Here is the answer to your question: {question}

    Helpful Answer (in Portuguese):
    """
}

print(f"Defined {len(prompt_templates)} prompt templates.")

Defined 2 prompt templates.


In [8]:
# chatbot function & experiments
def run_chatbot_experiment(prompt_template, question):
    """Runs a single query against the RAG chain and returns the result."""
    llm = AzureChatOpenAI(
        deployment_name=os.getenv("AZURE_DEPLOYMENT_NAME"),
        openai_api_version="2024-02-15-preview"
    )

    prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt}
    )

    return qa_chain({"query": question})

# --- Let's run our experiments! ---
test_questions = [
    "Posso construir uma cave para habitação?",
    "Qual é a altura mínima para o pé-direito de uma loja comercial?",
    "Qual a largura mínima de uma escada num prédio com 4 apartamentos?"
]

for prompt_name, prompt_text in prompt_templates.items():
    with mlflow.start_run(run_name=f"prompt_test_{prompt_name}"):
        print(f"\n--- Testing Prompt: {prompt_name} ---")
        mlflow.log_param("prompt_name", prompt_name)
        mlflow.log_text(prompt_text, "prompt.txt")

        for i, question in enumerate(test_questions):
            mlflow.log_param(f"question_{i}", question)

            result = run_chatbot_experiment(prompt_text, question)
            answer = result['result']

            # Log the answer as a text artifact
            mlflow.log_text(answer, f"answer_{i}.txt")
            print(f"Q: {question}")
            print(f"A: {answer}\n")

print("✅ All experiments completed. Check the MLflow UI at http://localhost:5001")


--- Testing Prompt: legal_expert ---


/tmp/ipykernel_558/2505925710.py:19: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return qa_chain({"query": question})


Q: Posso construir uma cave para habitação?
A: Não, uma cave só poderá ser utilizada para habitação se cumprir todas as condições de salubridade previstas no regulamento, além de atender requisitos como possuir pelo menos uma parede exterior completamente desafogada a partir de 0,15 m abaixo do nível do pavimento interior e adotar medidas contra infiltrações e humidade. Fora dessas condições, as caves são permitidas apenas como arrecadação ou armazém, sem comunicação direta com a parte habitacional do prédio (Artigos 78.º e 79.º).

Q: Qual é a altura mínima para o pé-direito de uma loja comercial?
A: O pé-direito mínimo para uma loja comercial é de 2,70 metros, conforme mencionado no Artigo 49.º.

Q: Qual a largura mínima de uma escada num prédio com 4 apartamentos?
A: A largura mínima da escada num prédio com 4 apartamentos é de 1,10 metros, conforme o artigo 10.º, n.º 3.

🏃 View run prompt_test_legal_expert at: http://mlflow:5000/#/experiments/1/runs/ec9be7bcdc454046999071518bb3a707
